# Data cleaning

Reads official `data/train.csv` and `data/test.csv`, applies **training-only** imputation, and writes:

- `data/train_cleaned.csv`
- `data/test_cleaned.csv`

EDA (`EDA.ipynb`) and modelling (`Modelling.ipynb`, `temp.py`, `fixed_model.py`) use these cleaned tables.

**Protocol.** `current_PM2_5` is imputed on **train only**. It is never created or filled on test.


In [1]:
from pathlib import Path
import pandas as pd

DATA = Path("data")
TARGET = "PM2_5_next_hour"
SENSOR_COLS = ["PM10", "SO2", "NO2", "CO", "O3", "TEMP", "PRES", "DEWP", "RAIN", "WSPM"]

train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
if "current_PM2_5" in test.columns:
    test = test.drop(columns=["current_PM2_5"])

print("train", train.shape, "test", test.shape)
print("test has current_PM2_5:", "current_PM2_5" in test.columns)
print("test has target:", TARGET in test.columns)


train (360954, 20) test (51063, 18)
test has current_PM2_5: False
test has target: False


/Users/yo/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## Rules

1. Parse `observation_timestamp`.
2. Wind direction `wd`: missing → `"Missing"`.
3. Numeric sensors (PM10, gases, weather): fill with **train-only column means** (same means applied to test).
4. `current_PM2_5`: fill with the **train** mean, on train rows only.
5. Do not drop rows. Target is already complete in train.
6. Do not add `current_PM2_5` or the target to test.


In [2]:
train["observation_timestamp"] = pd.to_datetime(train["observation_timestamp"], errors="raise")
test["observation_timestamp"] = pd.to_datetime(test["observation_timestamp"], errors="raise")

print("train duplicates", int(train.duplicated().sum()), "station-time", int(train.duplicated(["station", "observation_timestamp"]).sum()))
print("test  duplicates", int(test.duplicated().sum()), "station-time", int(test.duplicated(["station", "observation_timestamp"]).sum()))

print("\nMissing % before cleaning (train):")
print((train.isna().mean() * 100).sort_values(ascending=False).round(2).head(12))
print("\nMissing % before cleaning (test):")
print((test.isna().mean() * 100).sort_values(ascending=False).round(2).head(12))


train duplicates 0 station-time 0
test  duplicates 0 station-time 0

Missing % before cleaning (train):
CO               4.39
O3               2.35
NO2              2.08
SO2              1.29
current_PM2_5    0.68
PM10             0.53
wd               0.22
DEWP             0.05
PRES             0.05
TEMP             0.05
WSPM             0.05
RAIN             0.05
dtype: float64

Missing % before cleaning (test):
O3                       2.00
wd                       1.97
CO                       1.51
NO2                      1.31
SO2                      0.92
PM10                     0.58
DEWP                     0.43
TEMP                     0.43
RAIN                     0.42
PRES                     0.42
WSPM                     0.27
observation_timestamp    0.00
dtype: float64


In [3]:
train["wd"] = train["wd"].fillna("Missing")
test["wd"] = test["wd"].fillna("Missing")

train_sensor_means = train[SENSOR_COLS].mean()
train[SENSOR_COLS] = train[SENSOR_COLS].fillna(train_sensor_means)
test[SENSOR_COLS] = test[SENSOR_COLS].fillna(train_sensor_means)

if "current_PM2_5" in train.columns:
    train["current_PM2_5"] = train["current_PM2_5"].fillna(train["current_PM2_5"].mean())

print("wd filled with 'Missing'")
print("sensor means (train-only):")
display(train_sensor_means.to_frame("mean"))
print("remaining NA train", int(train.isna().sum().sum()), "test", int(test.isna().sum().sum()))


wd filled with 'Missing'
sensor means (train-only):
             mean
PM10   103.524493
SO2     16.375999
NO2     49.487934
CO    1184.508325
O3      60.843154
TEMP    14.472746
PRES  1009.756625
DEWP     3.282668
RAIN     0.066771
WSPM     1.734617
remaining NA train 0 test 0


In [4]:
assert TARGET in train.columns
assert train[TARGET].isna().sum() == 0
assert TARGET not in test.columns
assert "current_PM2_5" not in test.columns
assert train.isna().sum().sum() == 0
assert test.isna().sum().sum() == 0

train.to_csv(DATA / "train_cleaned.csv", index=False)
test.to_csv(DATA / "test_cleaned.csv", index=False)
print("Wrote", DATA / "train_cleaned.csv", train.shape)
print("Wrote", DATA / "test_cleaned.csv", test.shape)


Wrote data/train_cleaned.csv (360954, 20)
Wrote data/test_cleaned.csv (51063, 18)
